# Графический подход к множественному тестированию в A/B-экспериментаъ

## 1. Введение и напоминание базовых концепций

### 1.1 Краткое напоминание: FWER, FDR и классические поправки

Прежде чем мы погрузимся в графический подход, давайте освежим в памяти тот фундамент, на котором всё строится. Вы уже знакомы с проблемой множественных сравнений и основными методами её решения, поэтому быстро пройдёмся по ключевым понятиям, но обязательно выделим те ограничения классических процедур, которые как раз и мотивировали создание графического подхода.

Итак, представим, что в рамках одного A/B‑эксперимента или клинического исследования мы проверяем не одну, а сразу несколько нулевых гипотез: скажем, сравниваем несколько вариантов продукта по нескольким метрикам, или анализируем несколько конечных точек эффективности и безопасности. Классический пример из области, которую мы будем рассматривать, — клиническое исследование с двумя дозами препарата, тремя иерархически упорядоченными конечными точками, итого шесть элементарных гипотез. Или, что ближе вам, A/B‑тест с контролем и тремя вариантами, где у каждого варианта есть основная метрика — выручка, и несколько guadrail метрик, например, CTR и retention. Возникает вопрос: как нам принять решение по всему этому набору гипотез, сохраняя контроль над ошибками?

Проблема хорошо известна: если каждую из $m$ независимых нулевых гипотез проверять на привычном уровне значимости $\alpha$ (например, $0.05$), то вероятность совершить хотя бы одну ошибку I рода стремительно растёт. Формально:

$$
P(\text{хотя бы одна ошибка I рода}) = 1 - (1 - \alpha)^m\;.
$$

При $m = 10$ и $\alpha = 0.05$ эта вероятность уже около $0.40$, при $m = 20$ — порядка $0.64$. Иными словами, мы почти наверняка найдём какое‑то «значимое» различие просто в силу случайности. Поэтому нам нужны специальные метрики и процедуры, которые удерживают общую вероятность ложных выводов в заданных рамках.

Центральных метрики три.

1. **Групповая вероятность ошибки I рода**, Family‑Wise Error Rate (FWER)

$$
FWER = P(V \ge 1)\;, \text{ где V — число ложноположительных результатов}
$$

Контроль FWER на уровне $\alpha$ означает, что при любом раскладе истинных и ложных гипотез вероятность допустить хотя бы одну такую ошибку не превышает $\alpha$. Этот критерий очень строг, и именно его требуют регуляторы для подтверждающих клинических испытаний, потому что главная цель — минимизировать риск пропуска на рынок неэффективного или небезопасного препарата.

2. **Доля ложных отклонений**, False Discovery Rate (FDR)

$$
FDR = \mathbb{E}\!\left[\frac{V}{\max(R,1)}\right]\;, \text{ где $R$ — общее число отвергнутых гипотез}
$$

Фактически FDR показывает, какую долю среди всех наших «находок» в среднем составляют ложные. Контроль на уровне, скажем, $0.05$ допускает, что в среднем 5% объявленных значимыми результатов будут ошибочными. Это более мягкий критерий, чем FWER, и он особенно популярен в задачах с тысячами гипотез, например, в геномике или нейровизуализации. В контексте A/B‑тестов он тоже может применяться на разведочных стадиях.

3. **Обобщённая FWER**, $k$‑FWER:

$$
k\text{-}FWER = P(V \ge k)\;.
$$

Она контролирует вероятность совершить $k$ или более ложных отклонений. При $k=1$ это обычный FWER. Увеличивая $k$, мы ослабляем контроль, но можем повысить мощность — это полезно, когда мы готовы мириться с одним‑двумя лишними ложными сигналами ради обнаружения большего числа реальных эффектов.

Теперь вспомним три классические процедуры, которые вы уже знаете. Все они оперируют упорядоченными p‑значениями: $p_{(1)} \le p_{(2)} \le \dots \le p_{(m)}$.

1. **Поправка Бонферрони** — отвергаем любую гипотезу $H_i$, для которой $p_i \le \alpha / m$. Она гарантирует $FWER \le \alpha$ без всяких предположений о зависимости p‑значений, но платит за это чрезвычайной консервативностью.

2. **Процедура Холма**: мы идём от самого маленького p‑значения к самому большому и сравниваем $p_{(i)}$ с порогом $\alpha/(m-i+1)$. Как только встречаем p‑значение, которое больше своего порога, останавливаемся и отвергаем только предыдущие. Процедура Холма также контролирует FWER без каких‑либо предположений о структуре зависимости и является более мощной, чем Бонферрони

3. **Процедура Бенджамини–Хохберг**: алгоритм почти такой же, но ищется наибольшее $k$, для которого $p_{(k)} \le \frac{k}{m}\,\alpha$, и отвергаются все гипотезы с индексами от $1$ до $k$. Эта процедура гарантировано контролирует FDR только при независимости p‑значений или при положительной регрессионной зависимости.


На первый взгляд, этого арсенала достаточно. Однако давайте задумаемся: все эти процедуры обращаются с гипотезами **симметрично** — каждая гипотеза изначально рассматривается как равноправная. Но в реальных задачах, особенно в клинических испытаниях и сложных A/B‑тестах, гипотезы почти никогда не равноправны. У нас есть первичные конечные точки, которые обязательно должны быть значимы, чтобы исследование считалось успешным, и вторичные, которые мы тестируем только при условии успеха первичных. Есть иерархия важности, есть логические связи: например, мы не будем проверять безопасность дозировки, если не доказана её эффективность. Или в A/B‑тестировании: сначала нужно убедиться, что новый интерфейс не уронил ключевую метрику продаж, а затем уже смотреть на вспомогательные метрики вовлечённости.

Ни Бонферрони, ни Холм, ни BH не позволяют естественным образом отразить такую логику. Они не знают, что уровень значимости, «сэкономленный» на отвержении какой‑то вторичной гипотезы, должен быть передан обратно в первичное семейство, или что начальные права на проверку должны быть распределены неравномерно. В результате исследователи либо вынуждены применять кустарные, не вполне обоснованные схемы, либо теряют мощность там, где она критически важна.

Именно для преодоления этих ограничений Франк Бретц и его коллеги в 2009 году предложили **графический подход**. Ключевая идея — представить всё семейство гипотез в виде ориентированного взвешенного графа, где:

* вершины (узлы) — это элементарные нулевые гипотезы $H_i$;
* каждой вершине приписан начальный вес $\alpha_i$, причём сумма всех $\alpha_i$ не превышает общий уровень значимости $\alpha$;
* направленные рёбра от $H_i$ к $H_j$ имеют вес $g_{ij}$, который показывает, какая доля локального уровня $\alpha_i$ будет передана гипотезе $H_j$ после того, как $H_i$ отвергнута.

Таким образом, при отвержении гипотезы её уровень не пропадает, а **перераспределяется** по заданным правилам на оставшиеся, ещё не проверенные гипотезы. Это даёт возможность построить **последовательно отвергающую процедуру**, которая:

1. строго контролирует FWER на уровне $\alpha$;
2. учитывает приоритеты и логическую иерархию между гипотезами;
3. часто оказывается мощнее классических аналогов именно за счёт разумного «рециклинга» уровня значимости;
4. и, что немаловажно, представляет сложную стратегию тестирования в виде интуитивно понятной картинки, которую легко обсуждать с коллегами и регуляторами.

В последующих разделах мы формализуем этот подход, изучим его связь с замкнутой процедурой тестирования и увидим, как простые изменения в топологии графа превращают его из процедуры Бонферрони в процедуру Холма, затем в различные варианты гейткипинга и даже в процедуры, контролирующие $k$‑FWER. А сейчас, прежде чем перейти к мотивации на конкретных примерах, я зафиксирую главный тезис: **классические методы не умеют работать с иерархией гипотез; графический подход решает именно эту проблему, сохраняя строгий контроль над ошибками**.

### 1.2 Мотивация: от одного теста к сложному A/B-эксперименту

**Типичный A/B‑тест: больше не «один вариант — одна метрика»**

Ещё несколько лет назад типовой онлайн‑эксперимент выглядел так: у нас есть контрольная группа и один экспериментальный вариант, одна ключевая метрика успеха (скажем, конверсия в покупку) и несколько «охранных» метрик, которые мы проверяем на всякий случай. Сегодня же ландшафт гораздо сложнее. Рассмотрим несколько примеров, которые я буду использовать на протяжении всей лекции.

*Пример 1. Три варианта с иерархией метрик.*  
Предположим, продуктовая команда разработала три альтернативных интерфейса корзины: вариант A, вариант B и вариант C. Основная цель — увеличить средний чек (revenue per user). Одновременно нам критически важно не ухудшить пользовательский опыт: мы отслеживаем CTR на главную страницу, удержание (retention) через неделю и, возможно, время до первого заказа. Логика принятия решения такова:

1. Мы тестируем гипотезу о том, что вариант A превосходит контроль по revenue.  
2. Если это удалось доказать, только тогда мы проверяем, что A не ухудшает CTR, retention и время до заказа.  
3. Если же по revenue вариант A не значим, мы переходим к сравнению варианта B с контролем по revenue, и так далее.

Здесь мы имеем не плоское множество гипотез, а **двухуровневую иерархию**:
- **Первичный уровень:** три гипотезы о превосходстве по revenue для каждого варианта.
- **Вторичный уровень:** под каждой из этих гипотез висит семейство проверок на «не хуже» по вспомогательным метрикам.

*Пример 2. Клиническое исследование с множеством доз и конечных точек.*  
В статье Бретца и соавторов (2009) [1], а также в кейсах второй статьи [2], рассматривается поздняя фаза разработки препарата для лечения рассеянного склероза. Задача: сравнить две дозы нового препарата с плацебо по трём иерархически упорядоченным конечным точкам: частота рецидивов, число очагов в мозге на МРТ и прогрессирование инвалидизации. Итого шесть элементарных гипотез \(H_{11}, H_{12}, H_{13}\) (высокая доза по трём endpoint’ам) и \(H_{21}, H_{22}, H_{23}\) (низкая доза по тем же endpoint’ам). Клиническая логика требует:

- Если высокая доза не показала превосходства по первой, самой важной конечной точке, дальнейшее тестирование низкой дозы возможно, но с пониженным весом.
- Успех по первой точке для высокой дозы «открывает ворота» к проверке второй точки, а затем третьей — в строгой последовательности.

*Пример 3. Многошаговый протокол «K из N».*  
В исследовании Pre‑RELAX‑AHF (разобранном во второй статье [2]) требовалось доказать эффект экспериментального лечения по крайней мере на 6 из 9 заранее заданных конечных точек, связанных с острой сердечной недостаточностью. Лишь после выполнения этого условия разрешалось тестировать вторичные гипотезы безопасности. Здесь возникает ещё более сложная логическая связь: первичный успех определяется не одной гипотезой, а конъюнкцией («хотя бы шесть из девяти»).

**К чему приводит отсутствие учёта иерархии?**

Если мы воспользуемся классической процедурой Холма для всех гипотез одновременно, то каждый отдельный тест получит долю \(\alpha\), пропорциональную общему числу гипотез, и порядок проверки будет диктоваться исключительно величиной p‑значений. Это порождает две серьёзные проблемы:

1. **Неэффективное распределение \(\alpha\).** Предположим, в примере с вариантами интерфейса самый маленький p‑value получился у варианта C по revenue, а следующий по величине — у варианта A по вспомогательной метрике. Процедура Холма потратит часть общего уровня \(\alpha\) на проверку вспомогательной метрики варианта A, хотя вариант A ещё не доказал превосходства по основной метрике! Это противоречит бизнес‑логике: мы не должны «тратить» статистический бюджет на проверку безопасности того, что не доказало эффективность.

2. **Риск преждевременной остановки.** Представьте себе фиксированную последовательность тестов (fixed sequence test) [1, раздел 2.3], где все гипотезы выстроены в одну цепочку \(H_{11} \to H_{21} \to H_{12} \to \dots \to H_{23}\). Если первая гипотеза \(H_{11}\) (высокая доза vs плацебо по relapse rate) не отвергается на уровне \(\alpha\), процедура немедленно останавливается, и все остальные гипотезы, даже с очень маленькими p‑значениями, формально не могут быть отвергнуты. Исследование признаётся неуспешным полностью, хотя низкая доза, возможно, была эффективной. Жёсткая фиксация порядка тестирования, таким образом, приводит к неприемлемой потере мощности.

**Идея графического представления**

Графический подход [1] преодолевает эти ограничения, предлагая мыслить о семействе гипотез как о **взвешенном ориентированном графе**, где рёбра несут в себе правила «наследования» уровня значимости. Вернёмся к примеру с двумя дозами и тремя endpoint’ами. Мы можем построить граф, в котором:

- Вершинам, соответствующим гипотезам эффективности первой (самой важной) конечной точки, присваиваются наибольшие начальные веса.
- Рёбра направлены от первичных гипотез к вторичным и от более важных endpoint’ов к менее важным.
- Веса рёбер определяют, как перераспределяется уровень \(\alpha\) после отвержения той или иной гипотезы.

Например, мы можем договориться, что \(\alpha\) распределяется поровну между двумя дозами, и внутри каждой дозы идёт фиксированная последовательность от первой конечной точки к третьей. Но в отличие от жёсткой фиксированной последовательности, если высокая доза «провалится», её доля \(\alpha/2\) не пропадёт — она перетечёт на гипотезы низкой дозы! И наоборот, если высокая доза окажется успешной по всем трём endpoint’ам, освободившийся уровень может быть передан для усиления тестов низкой дозы. Эта гибкая система «рециклинга» значимости в точности отражает иерархическую логику принятия решений.

**Метафора «бюджета»**

Позвольте мне ввести интуитивную метафору, которая поможет нам мыслить о графическом подходе на протяжении всей лекции. Представьте, что общий уровень значимости \(\alpha\) — это бюджет, который спонсор исследования выделяет на проверку всех гипотез. Каждая вершина графа — это исследовательская группа, которой выделена определённая часть бюджета \((\alpha_i)\). Когда группа успешно завершает свою работу (гипотеза отвергается), она не кладёт оставшиеся деньги себе в карман, а переводит их следующей группе согласно заранее составленному плану (рёбра с весами \(g_{ij}\)). Более того, если какая‑то группа не справилась (гипотеза не отвергнута), она просто остаётся без финансирования, а её бюджет уже учтён в начальном распределении.

**Преимущества графического подхода — анонс**

Графический подход обладает тремя ключевыми достоинствами, которые делают его привлекательным и для клинических испытаний, и для онлайн‑экспериментов:

1. **Строгий контроль FWER.** Несмотря на всю гибкость, процедура гарантирует, что вероятность хотя бы одного ложного отклонения не превышает заданного \(\alpha\). Математически это опирается на замкнутую процедуру тестирования, с которой мы познакомимся в разделе 2.3.

2. **Прозрачность и коммуницируемость.** Граф можно нарисовать на доске и обсудить с клинической командой или с продакт‑менеджерами. Им не нужно вникать в дебри длинных таблиц весов для всех \(2^m - 1\) пересечений гипотез. Связи и приоритеты сразу видны.

3. **Гибкость и мощность.** За счёт перераспределения уровня значимости графические процедуры почти всегда мощнее или не уступают своим классическим аналогам, особенно когда гипотезы упорядочены. В дальнейшем мы увидим, как можно донастраивать графы с помощью \(\varepsilon\)-рёбер и расширений на случай контроля \(k\)-FWER и FDP (об этом пойдёт речь во второй части лекции, основанной на статье [2]).

---

**Что дальше?**

Прежде чем переходить к формальному описанию алгоритмов, давайте зафиксируем главный тезис этого блока: **в сложных A/B‑тестах и клинических исследованиях гипотезы образуют естественную иерархию; графический подход позволяет перенести эту иерархию на граф и затем автоматически построить корректную процедуру множественного тестирования, контролирующую FWER.** В следующем разделе мы наденем математические очки и строго определим все компоненты: вершины, веса, матрицу переходов и сам алгоритм.

## 2. Формальный аппарат графического подхода

### 2.1 Граф как математическая модель семейства гипотез
Определения: вершины (элементарные гипотезы), начальный вектор локальных уровней значимости **α**, матрица переходов **G**.  
Правила ограничений на веса. Визуализация графа на примерах (Рис. 1, 2 из статьи [1]).

In [ ]:
# TODO: Импорты библиотек (numpy, networkx, matplotlib)
# Создание первого демонстрационного графа (например, полносвязный с равными весами)

### 2.2 Алгоритм последовательного тестирования
Детальное описание Алгоритма 1 из статьи [1] (инициализация, выбор гипотезы, проверка, обновление графа).  
Повторение шагов до остановки. Свойства: сильный контроль FWER, shortcut, независимость от порядка удаления.

In [ ]:
# TODO: Псевдокод / заготовка функции graphical_test

### 2.3 Связь с замкнутой процедурой тестирования
Определение замкнутой процедуры тестирования (closed testing procedure).  
Как графический алгоритм порождает веса для всех пересечений гипотез и удовлетворяет условию монотонности.  
Доказательство того, что графический подход – это shortcut замкнутого теста, гарантирующий контроль FWER (Appendix A в [1]).

## 3. Детальный разбор ключевых примеров

### 3.1 Пример 0: поправка Бонферрони как вырожденный граф
Граф без рёбер: начальные веса α/m, все g_ij = 0. Демонстрация того, что это приводит к обычной поправке Бонферрони.

In [ ]:
# TODO: Настройка графа Бонферрони, запуск теста, вывод результата

### 3.2 Пример 1: процедура Холма как граф
Построение полносвязного графа с равными весами переходов. Пошаговая анимация/вывод состояний для заданных p-значений (Рис. 4 из [1]).

In [ ]:
# TODO: Реализация процедуры Холма через графический алгоритм
# Визуализация начального, промежуточных и конечного графа

### 3.3 Пример 2: gatekeeping процедура для двух семейств
Первичные гипотезы с весами α/2, вторичные с нулевыми начальными весами. Показать оригинальную и улучшенную версию с ε-рёбрами (Рис. 12 из [1]).

In [ ]:
# TODO: Класс/функция, поддерживающая ε-веса; пример gatekeeping

### 3.4 Интерактивная визуализация и код
Объединение примеров в единый класс `Graph`. Возможность менять p-значения и топологию графа в виджетах (если позволяет время).  
Вывод текущих весов вершин и матрицы переходов после каждого шага.

In [ ]:
# TODO: Интерактивные элементы (ipywidgets) для демонстрации работы алгоритма в реальном времени

## 4. Расширения графического подхода для обобщённых ошибок

### 4.1 Контроль k-FWER и FDP
Мотивация использования k-FWER и FDP в разведочных A/B-тестах.  
Описание дополненной (augmented) и обобщённой (generalised) процедур (Статья [2], Алгоритмы 1, 3, 4, 6).  
Сравнение их мощности и ограничений на примере фармакодинамического исследования (раздел 6.1).

In [ ]:
# TODO: Реализация augmented_k_FWER и augmented_FDP, загрузка данных кейса из статьи [2]

### 4.2 Адаптация к динамическим A/B-тестам
Примеры многошаговых протоколов: Pre-RELAX-AHF (6-из-9 критерий) и ATMOSPHERE (иерархия из трёх первичных и двух вторичных семейств).  
Демонстрация построения графа и применения дополненных процедур для этих кейсов.

In [ ]:
# TODO: Настройка графа для ATMOSPHERE, вызов k-FWER/FDP процедуры

## 5. Программная реализация на Python

### 5.1 Инструменты и библиотеки
Обзор используемых Python-библиотек: `numpy`, `networkx`, `matplotlib`.  
Структура кода: класс `Graph` и функция `graphical_test`. Пример реализации Алгоритма 1 из [1].

In [ ]:
# TODO: Полный код базового графического теста (уже подготовлен, но показать ключевые моменты)

### 5.2 Реализация дополненных процедур для k-FWER и FDP на Python
Функции `augmented_k_FWER` и `augmented_FDP`. Демонстрация работы на данных из кейсов.  
Замечание о вычислительной сложности и упрощённых (streamlined) версиях.

In [ ]:
# TODO: Код функций augmented_k_FWER / augmented_FDP; сравнение результатов с обычным FWER

## 6. Сравнительный анализ и практические рекомендации

### 6.1 Когда выбирать графический подход
Критерии выбора: иерархическая структура гипотез, необходимость прозрачной коммуникации, потребность в «рециклинге» уровня значимости.  
Сравнение с методом Бенджамини–Хохберг и процедурами на основе ресемплинга.  
Ограничения, связанные с корреляцией p-значений.

### 6.2 «Подводные камни» и направления развития
Проблемы обобщённой процедуры (не передаёт уровень гипотезам с малым числом доноров), вычислительные трудности при больших m и k>1.  
Открытые вопросы: скорректированные p-значения для дополненных процедур, онлайн-мониторинг экспериментов.

## 7. Заключение и Q&A

Резюме основных идей. Домашнее задание: воспроизвести примеры и построить граф для гипотетического A/B-теста с тремя вариантами и двумя метриками.  
Ответы на вопросы.

In [ ]:
# Место для дополнительных интерактивных вопросов или демонстраций